In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

# Read all train CSVs — SMD has no header, 38 columns (metrics)
train_df = (spark.read
  .option("header", "false")
  .option("inferSchema", "true")
  .csv("/Volumes/workspace/default/dataset/smd/train/train/*.txt")
  .toDF(*[f"metric_{i}" for i in range(38)])
  .withColumn("source_file",
      F.regexp_extract(F.col("_metadata.file_path"), r"(machine-\d+-\d+)", 1))
  .withColumn("ingested_at", F.current_timestamp())
)

# Write to Bronze Delta table
(train_df.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("/Workspace/Users/nadhiya.ganesan93@gmail.com/.assistant/smd/smd_bronze_train")
)

print(f"Rows ingested: {train_df.count():,}")
train_df.printSchema()

---------------------------------------------------------------------------
ParseException                            Traceback (most recent call last)
File <command-4948507986249800>, line 19
      5 train_df = (spark.read
      6   .option("header", "false")
      7   .option("inferSchema", "true")
   (...)
     12   .withColumn("ingested_at", F.current_timestamp())
     13 )
     15 # Write to Bronze Delta table
     16 (train_df.write
     17   .format("delta")
     18   .mode("overwrite")
---> 19   .saveAsTable("/Workspace/Users/nadhiya.ganesan93@gmail.com/.assistant/smd/smd_bronze_train")
     20 )
     22 print(f"Rows ingested: {train_df.count():,}")
     23 train_df.printSchema()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.clie

In [0]:
import os
print(os.listdir("/Volumes/workspace/default/dataset/smd/train/train"))

['machine-1-1.txt', 'machine-1-2.txt', 'machine-1-3.txt', 'machine-1-4.txt', 'machine-1-5.txt', 'machine-1-6.txt', 'machine-1-7.txt', 'machine-1-8.txt', 'machine-2-1.txt', 'machine-2-2.txt', 'machine-2-3.txt', 'machine-2-4.txt', 'machine-2-5.txt', 'machine-2-6.txt', 'machine-2-7.txt', 'machine-2-8.txt', 'machine-2-9.txt', 'machine-3-1.txt', 'machine-3-10.txt', 'machine-3-11.txt', 'machine-3-2.txt', 'machine-3-3.txt', 'machine-3-4.txt', 'machine-3-5.txt', 'machine-3-6.txt', 'machine-3-7.txt', 'machine-3-8.txt', 'machine-3-9.txt']


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read all train .txt files from Volume
train_df = (spark.read
  .option("header", "false")
  .option("inferSchema", "true")
  .csv("/Volumes/workspace/default/dataset/smd/train/train/*.txt")
  .toDF(*[f"metric_{i}" for i in range(38)])
 .withColumn("source_file",
      F.regexp_extract(F.col("_metadata.file_path"), r"(machine-\d+-\d+)", 1))
  .withColumn("ingested_at", F.current_timestamp())
)

(train_df.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("workspace.default.smd_bronze_train")
)

print(f"✅ Rows ingested: {train_df.count():,}")
print(f"   Machines found: {train_df.select('source_file').distinct().count()}")
train_df.show(3)

✅ Rows ingested: 708,405
   Machines found: 28
+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-----------+--------------------+
|metric_0|metric_1|metric_2|metric_3|metric_4|metric_5|metric_6|metric_7|metric_8|metric_9|metric_10|metric_11|metric_12|metric_13|metric_14|metric_15|metric_16|metric_17|metric_18|metric_19|metric_20|metric_21|metric_22|metric_23|metric_24|metric_25|metric_26|metric_27|metric_28|metric_29|metric_30|metric_31|metric_32|metric_33|metric_34|metric_35|metric_36|metric_37|source_file|         ingested_at|
+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+--

In [0]:
df = spark.table("workspace.default.smd_bronze_train")
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.groupBy("source_file").count().orderBy("source_file").show(30)

Rows: 708,405
Columns: 40
+------------+-----+
| source_file|count|
+------------+-----+
| machine-1-1|28479|
| machine-1-2|23694|
| machine-1-3|23702|
| machine-1-4|23706|
| machine-1-5|23705|
| machine-1-6|23688|
| machine-1-7|23697|
| machine-1-8|23698|
| machine-2-1|23693|
| machine-2-2|23699|
| machine-2-3|23688|
| machine-2-4|23689|
| machine-2-5|23688|
| machine-2-6|28743|
| machine-2-7|23696|
| machine-2-8|23702|
| machine-2-9|28722|
| machine-3-1|28700|
|machine-3-10|23692|
|machine-3-11|28695|
| machine-3-2|23702|
| machine-3-3|23703|
| machine-3-4|23687|
| machine-3-5|23690|
| machine-3-6|28726|
| machine-3-7|28705|
| machine-3-8|28703|
| machine-3-9|28713|
+------------+-----+



In [0]:
df = spark.table("workspace.default.smd_bronze_train")
print(f"Rows: {df.count():,}")
df.show(3)

Rows: 708,405
+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-----------+--------------------+
|metric_0|metric_1|metric_2|metric_3|metric_4|metric_5|metric_6|metric_7|metric_8|metric_9|metric_10|metric_11|metric_12|metric_13|metric_14|metric_15|metric_16|metric_17|metric_18|metric_19|metric_20|metric_21|metric_22|metric_23|metric_24|metric_25|metric_26|metric_27|metric_28|metric_29|metric_30|metric_31|metric_32|metric_33|metric_34|metric_35|metric_36|metric_37|source_file|         ingested_at|
+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+---------+---------+---------+---------+---------+---------+---------+---------+-----

Cell 1 — Clean and add row index


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Window per machine ordered by position
w_machine = Window.partitionBy("source_file").orderBy(F.monotonically_increasing_id())

silver_df = (spark.table("workspace.default.smd_bronze_train")
  # Fill nulls with 0 (metrics are normalised 0-1 so 0 is safe)
  .fillna(0.0)
  # Add row number per machine — this is our time step
  .withColumn("time_step", F.row_number().over(w_machine))
)

print(f"Rows: {silver_df.count():,}")
silver_df.select("source_file", "time_step", "metric_0", "metric_1").show(5)

Rows: 708,405
+-----------+---------+--------+--------+
|source_file|time_step|metric_0|metric_1|
+-----------+---------+--------+--------+
|machine-2-2|        1|     0.0|  5.1E-4|
|machine-2-2|        2|     0.0| 5.95E-4|
|machine-2-2|        3|     0.0|0.001445|
|machine-2-2|        4|     0.0| 0.00102|
|machine-2-2|        5|     0.0|  3.4E-4|
+-----------+---------+--------+--------+
only showing top 5 rows


Cell 2 — Add rolling window features


In [0]:
# Rolling windows of 5, 15, 60 time steps per machine
# We'll do this on metric_0 (CPU proxy) to start — same pattern applies to all

w5  = Window.partitionBy("source_file").orderBy("time_step").rowsBetween(-4, 0)
w15 = Window.partitionBy("source_file").orderBy("time_step").rowsBetween(-14, 0)
w60 = Window.partitionBy("source_file").orderBy("time_step").rowsBetween(-59, 0)

silver_df = (silver_df
  # Rolling mean — is the metric trending up?
  .withColumn("metric_0_mean_5",  F.mean("metric_0").over(w5))
  .withColumn("metric_0_mean_15", F.mean("metric_0").over(w15))
  .withColumn("metric_0_mean_60", F.mean("metric_0").over(w60))
  # Rolling std — is the metric becoming unstable?
  .withColumn("metric_0_std_5",   F.stddev("metric_0").over(w5))
  .withColumn("metric_0_std_15",  F.stddev("metric_0").over(w15))
  # Rate of change — how fast is it spiking?
  .withColumn("metric_0_lag1",    F.lag("metric_0", 1).over(
                                    Window.partitionBy("source_file").orderBy("time_step")))
  .withColumn("metric_0_roc",     F.col("metric_0") - F.col("metric_0_lag1"))
  .fillna(0.0)  # fill nulls created by lag at start of each machine
)

print(f"Rows: {silver_df.count():,}")
silver_df.select("source_file", "time_step", "metric_0", 
                 "metric_0_mean_5", "metric_0_std_5", "metric_0_roc").show(5)

Rows: 708,405
+-----------+---------+--------+---------------+--------------+------------+
|source_file|time_step|metric_0|metric_0_mean_5|metric_0_std_5|metric_0_roc|
+-----------+---------+--------+---------------+--------------+------------+
|machine-1-5|        1|0.010101|       0.010101|           0.0|         0.0|
|machine-1-5|        2|0.010101|       0.010101|           0.0|         0.0|
|machine-1-5|        3|0.010101|       0.010101|           0.0|         0.0|
|machine-1-5|        4|0.010101|       0.010101|           0.0|         0.0|
|machine-1-5|        5|0.010101|       0.010101|           0.0|         0.0|
+-----------+---------+--------+---------------+--------------+------------+
only showing top 5 rows


In [0]:
(silver_df.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("workspace.default.smd_silver_train")
)

print(f"✅ Silver table written")
print(f"Rows: {spark.table('workspace.default.smd_silver_train').count():,}")
print(f"Columns: {len(spark.table('workspace.default.smd_silver_train').columns)}")

✅ Silver table written
Rows: 708,405
Columns: 48


In [0]:
import os
print("TEST:", os.listdir("/Volumes/workspace/default/dataset/smd/test/"))
print("LABELS:", os.listdir("/Volumes/workspace/default/dataset/smd/test-label/"))

TEST: ['machine-1-1.txt', 'machine-1-2.txt', 'machine-1-3.txt', 'machine-1-4.txt', 'machine-1-5.txt', 'machine-1-6.txt', 'machine-1-7.txt', 'machine-1-8.txt', 'machine-2-1.txt', 'machine-2-2.txt', 'machine-2-3.txt', 'machine-2-4.txt', 'machine-2-5.txt', 'machine-2-6.txt', 'machine-2-7.txt', 'machine-2-8.txt', 'machine-2-9.txt', 'machine-3-1.txt', 'machine-3-10.txt', 'machine-3-11.txt', 'machine-3-2.txt', 'machine-3-3.txt', 'machine-3-4.txt', 'machine-3-5.txt', 'machine-3-6.txt', 'machine-3-7.txt', 'machine-3-8.txt', 'machine-3-9.txt']
LABELS: ['machine-1-1.txt', 'machine-1-2.txt', 'machine-1-3.txt', 'machine-1-4.txt', 'machine-1-5.txt', 'machine-1-6.txt', 'machine-1-7.txt', 'machine-1-8.txt', 'machine-2-1.txt', 'machine-2-2.txt', 'machine-2-3.txt', 'machine-2-4.txt', 'machine-2-5.txt', 'machine-2-6.txt', 'machine-2-7.txt', 'machine-2-8.txt', 'machine-2-9.txt', 'machine-3-1.txt', 'machine-3-10.txt', 'machine-3-11.txt', 'machine-3-2.txt', 'machine-3-3.txt', 'machine-3-4.txt', 'machine-3-

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load test data
test_df = (spark.read
  .option("header", "false")
  .option("inferSchema", "true")
  .csv("/Volumes/workspace/default/dataset/smd/test/*.txt")
  .toDF(*[f"metric_{i}" for i in range(38)])
  .withColumn("source_file",
      F.regexp_extract(F.col("_metadata.file_path"), r"(machine-\d+-\d+)", 1))
  .withColumn("ingested_at", F.current_timestamp())
)

# Load labels
label_df = (spark.read
  .option("header", "false")
  .csv("/Volumes/workspace/default/dataset/smd/test-label/*.txt")
  .toDF("is_anomaly")
  .withColumn("is_anomaly", F.col("is_anomaly").cast("integer"))
  .withColumn("source_file",
      F.regexp_extract(F.col("_metadata.file_path"), r"(machine-\d+-\d+)", 1))
)

# Join labels to test using row number per machine
w = Window.partitionBy("source_file").orderBy(F.monotonically_increasing_id())

test_labeled = (test_df
  .withColumn("row_id", F.row_number().over(w))
  .join(
    label_df.withColumn("row_id", F.row_number().over(w)),
    ["source_file", "row_id"]
  )
  .drop("row_id")
)

(test_labeled.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("workspace.default.smd_bronze_test")
)

print(f"✅ Test rows: {test_labeled.count():,}")
print(f"   Anomaly rate: {test_labeled.filter('is_anomaly=1').count() / test_labeled.count():.1%}")

✅ Test rows: 708,420
   Anomaly rate: 4.2%


In [0]:
w_machine = Window.partitionBy("source_file").orderBy(F.monotonically_increasing_id())
w5  = Window.partitionBy("source_file").orderBy("time_step").rowsBetween(-4, 0)
w15 = Window.partitionBy("source_file").orderBy("time_step").rowsBetween(-14, 0)
w60 = Window.partitionBy("source_file").orderBy("time_step").rowsBetween(-59, 0)

silver_test_df = (spark.table("workspace.default.smd_bronze_test")
  .fillna(0.0)
  .withColumn("time_step", F.row_number().over(w_machine))
  .withColumn("metric_0_mean_5",  F.mean("metric_0").over(w5))
  .withColumn("metric_0_mean_15", F.mean("metric_0").over(w15))
  .withColumn("metric_0_mean_60", F.mean("metric_0").over(w60))
  .withColumn("metric_0_std_5",   F.stddev("metric_0").over(w5))
  .withColumn("metric_0_std_15",  F.stddev("metric_0").over(w15))
  .withColumn("metric_0_lag1",    F.lag("metric_0", 1).over(
                                    Window.partitionBy("source_file").orderBy("time_step")))
  .withColumn("metric_0_roc",     F.col("metric_0") - F.col("metric_0_lag1"))
  .fillna(0.0)
)

(silver_test_df.write
  .format("delta")
  .mode("overwrite")
  .saveAsTable("workspace.default.smd_silver_test")
)

print(f"✅ Silver test written: {silver_test_df.count():,} rows, {len(silver_test_df.columns)} cols")

✅ Silver test written: 708,420 rows, 49 cols


In [0]:
# Select final features for training — raw metrics + rolling features + label
feature_cols = [f"metric_{i}" for i in range(38)] + [
  "metric_0_mean_5", "metric_0_mean_15", "metric_0_mean_60",
  "metric_0_std_5",  "metric_0_std_15",  "metric_0_roc"
]

# Gold train — no label needed, unsupervised training
gold_train = (spark.table("workspace.default.smd_silver_train")
  .select(feature_cols + ["source_file", "time_step"])
)

# Gold test — includes label for evaluation
gold_test = (spark.table("workspace.default.smd_silver_test")
  .select(feature_cols + ["source_file", "time_step", "is_anomaly"])
)

(gold_train.write.format("delta").mode("overwrite")
  .saveAsTable("workspace.default.smd_gold_train"))

(gold_test.write.format("delta").mode("overwrite")
  .saveAsTable("workspace.default.smd_gold_test"))

print(f"✅ Gold train: {gold_train.count():,} rows, {len(gold_train.columns)} cols")
print(f"✅ Gold test:  {gold_test.count():,} rows, {len(gold_test.columns)} cols")
print(f"   Features:   {len(feature_cols)}")

✅ Gold train: 708,405 rows, 46 cols
✅ Gold test:  708,420 rows, 47 cols
   Features:   44


In [0]:
test_check = spark.table("workspace.default.smd_gold_test")
print("Anomaly distribution:")
test_check.groupBy("is_anomaly").count().show()

print("Sample Gold features:")
test_check.select("source_file", "metric_0", "metric_0_mean_5", 
                  "metric_0_std_5", "metric_0_roc", "is_anomaly").show(5)

Anomaly distribution:
+----------+------+
|is_anomaly| count|
+----------+------+
|         0|678976|
|         1| 29444|
+----------+------+

Sample Gold features:
+-----------+--------+-------------------+--------------------+--------------------+----------+
|source_file|metric_0|    metric_0_mean_5|      metric_0_std_5|        metric_0_roc|is_anomaly|
+-----------+--------+-------------------+--------------------+--------------------+----------+
|machine-1-1|0.075269|           0.075269|                 0.0|                 0.0|         0|
|machine-1-1|0.086022|0.08064550000000001|0.007603519218098945|0.010752999999999999|         0|
|machine-1-1|0.075269|0.07885333333333334|0.006208247444596047|-0.01075299999999...|         0|
|machine-1-1|0.086022|0.08064550000000001|0.006208247444596044|0.010752999999999999|         0|
|machine-1-1|0.086022|0.08172080000000001|0.005889660660853048|                 0.0|         0|
+-----------+--------+-------------------+--------------------+----

In [0]:
%pip install xgboost
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


Cell 1 — Convert Gold tables to pandas and prepare features


In [0]:
import pandas as pd
import numpy as np

# Convert to pandas for sklearn
train_pd = spark.table("workspace.default.smd_gold_train").toPandas()
test_pd  = spark.table("workspace.default.smd_gold_test").toPandas()

# Feature columns — everything except metadata
feature_cols = [f"metric_{i}" for i in range(38)] + [
  "metric_0_mean_5", "metric_0_mean_15", "metric_0_mean_60",
  "metric_0_std_5",  "metric_0_std_15",  "metric_0_roc"
]

X_train = train_pd[feature_cols]
X_test  = test_pd[feature_cols]
y_test  = test_pd["is_anomaly"]

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Anomalies in test: {y_test.sum():,} ({y_test.mean():.1%})")

X_train: (708405, 44)
X_test:  (708420, 44)
Anomalies in test: 29,444 (4.2%)


In [0]:
Cell 2 — Train XGBoost and log everything to MLflow


In [0]:
%pip install xgboost
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 MB 190.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.1/300.1 MB 147.0 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import mlflow.sklearn
from xgboost import XGBClassifier
from sklearn.metrics import (f1_score, roc_auc_score,
                             precision_score, recall_score,
                             classification_report)
from sklearn.model_selection import train_test_split

# Since SMD test set has labels, use it for supervised training
# Split test into train/eval — 70% train, 30% evaluate
X_train_sup, X_eval, y_train_sup, y_eval = train_test_split(
    X_test, y_test, test_size=0.3, random_state=42, stratify=y_test
)

# Class imbalance ratio
scale = int(len(y_train_sup[y_train_sup==0]) / len(y_train_sup[y_train_sup==1]))
print(f"Training rows:    {len(X_train_sup):,}")
print(f"Evaluation rows:  {len(X_eval):,}")
print(f"scale_pos_weight: {scale}")

mlflow.set_experiment("/Users/nadhiya.ganesan93@gmail.com/smd-anomaly-detection")

with mlflow.start_run(run_name="xgboost_supervised"):

    params = {
        "n_estimators": 200,
        "max_depth": 6,
        "learning_rate": 0.1,
        "scale_pos_weight": scale,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    }
    mlflow.log_params(params)

    # Train on labelled data
    model = XGBClassifier(**params, random_state=42, eval_metric="logloss")
    model.fit(X_train_sup, y_train_sup)

    # Evaluate
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]

    f1        = f1_score(y_eval, y_pred)
    auc       = roc_auc_score(y_eval, y_prob)
    precision = precision_score(y_eval, y_pred, zero_division=0)
    recall    = recall_score(y_eval, y_pred, zero_division=0)

    mlflow.log_metric("f1_score", f1)
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)

    # Log model with signature to avoid the warning
    input_example = X_eval.iloc[:5]
    mlflow.sklearn.log_model(
        model, 
        name="anomaly_detector",
        input_example=input_example
    )

    print(f"✅ Run logged to MLflow")
    print(f"   F1:        {f1:.3f}")
    print(f"   AUC:       {auc:.3f}")
    print(f"   Precision: {precision:.3f}")
    print(f"   Recall:    {recall:.3f}")
    print(f"\n{classification_report(y_eval, y_pred, target_names=['normal','anomaly'])}")

Training rows:    495,894
Evaluation rows:  212,526
scale_pos_weight: 23


🔗 View Logged Model at: https://dbc-fc3c1559-f225.cloud.databricks.com/ml/experiments/616568261592496/models/m-463af32bb18e41fe8b3de8f53e6b0a34?o=7474657727450520
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


✅ Run logged to MLflow
   F1:        0.718
   AUC:       0.992
   Precision: 0.577
   Recall:    0.950

              precision    recall  f1-score   support

      normal       1.00      0.97      0.98    203693
     anomaly       0.58      0.95      0.72      8833

    accuracy                           0.97    212526
   macro avg       0.79      0.96      0.85    212526
weighted avg       0.98      0.97      0.97    212526



In [0]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get the best run by F1 score
best_run = mlflow.search_runs(
    experiment_names=["/Users/nadhiya.ganesan93@gmail.com/smd-anomaly-detection"],
    order_by=["metrics.f1_score DESC"]
).iloc[0]

run_id    = best_run["run_id"]
model_uri = f"runs:/{run_id}/anomaly_detector"

print(f"Best run ID: {run_id}")
print(f"F1: {best_run['metrics.f1_score']:.3f}")
print(f"AUC: {best_run['metrics.auc']:.3f}")

# Register model
reg = mlflow.register_model(
    model_uri=model_uri,
    name="smd-anomaly-detector"
)

print(f"\n✅ Model registered!")
print(f"   Name:    {reg.name}")
print(f"   Version: {reg.version}")

Best run ID: 776802e1c03049fd8d8795f7f26e5cf2
F1: 0.718
AUC: 0.992


Successfully registered model 'workspace.default.smd-anomaly-detector'.
2026/06/07 17:23:34 WARNING mlflow.tracking._model_registry.fluent: Run with id 776802e1c03049fd8d8795f7f26e5cf2 has no artifacts at artifact path 'anomaly_detector', registering model based on models:/m-3aea7818690d4b45a1ca0cfcc161c8ba instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.smd-anomaly-detector': https://dbc-fc3c1559-f225.cloud.databricks.com/explore/data/models/workspace/default/smd-anomaly-detector/version/1?o=7474657727450520



✅ Model registered!
   Name:    workspace.default.smd-anomaly-detector
   Version: 1


In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Set alias to champion — marks this as the production model
client.set_registered_model_alias(
    name="workspace.default.smd-anomaly-detector",
    alias="champion",
    version=reg.version
)

# Confirm
model_info = client.get_registered_model("workspace.default.smd-anomaly-detector")
print(f"✅ Model registered and aliased")
print(f"   Name:    {model_info.name}")
print(f"   Alias:   champion → version {reg.version}")
print(f"\nYou can view it in Databricks:")
print(f"Catalog → workspace → default → smd-anomaly-detector")

✅ Model registered and aliased
   Name:    workspace.default.smd-anomaly-detector
   Alias:   champion → version 1

You can view it in Databricks:
Catalog → workspace → default → smd-anomaly-detector


In [0]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
model_info = client.get_registered_model("workspace.default.smd-anomaly-detector")
print(f"Model: {model_info.name}")

# Check alias
version_info = client.get_model_version_by_alias(
    name="workspace.default.smd-anomaly-detector",
    alias="champion"
)
print(f"Champion version: {version_info.version}")
print(f"Status: {version_info.status}")
print(f"\n✅ Ready for deployment")

Model: workspace.default.smd-anomaly-detector
Champion version: 1
Status: READY

✅ Ready for deployment


 Create a serving endpoint

In [0]:
import requests
import json

# Get your Databricks workspace URL and token
DATABRICKS_HOST = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Define the endpoint
endpoint_config = {
    "name": "smd-anomaly-detector-endpoint",
    "config": {
        "served_entities": [
            {
                "entity_name": "workspace.default.smd-anomaly-detector",
                "entity_version": "1",
                "workload_size": "Small",
                "scale_to_zero_enabled": True
            }
        ]
    }
}

# Create the endpoint
response = requests.post(
    f"{DATABRICKS_HOST}/api/2.0/serving-endpoints",
    headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}"},
    json=endpoint_config
)

print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

Status: 400
{
  "error_code": "RESOURCE_ALREADY_EXISTS",
  "message": "Endpoint with name 'smd-anomaly-detector-endpoint' already exists.",
  "details": [
    {
      "@type": "type.googleapis.com/google.rpc.RequestInfo",
      "request_id": "a239570d-e72f-46cf-bafb-b60104a3c5d7",
      "serving_data": ""
    }
  ]
}


In [0]:
import time

def check_endpoint_status():
    response = requests.get(
        f"{DATABRICKS_HOST}/api/2.0/serving-endpoints/smd-anomaly-detector-endpoint",
        headers={"Authorization": f"Bearer {DATABRICKS_TOKEN}"}
    )
    return response.json().get("state", {}).get("ready", "NOT_READY")

print("Waiting for endpoint to be ready...")
print("(This takes 5–10 minutes — normal for first deployment)\n")

for i in range(20):
    status = check_endpoint_status()
    print(f"  [{i+1}/20] Status: {status}")
    if status == "READY":
        print("\n✅ Endpoint is live!")
        break
    time.sleep(30)

Waiting for endpoint to be ready...
(This takes 5–10 minutes — normal for first deployment)

  [1/20] Status: READY

✅ Endpoint is live!


In [0]:
# Score 5 rows from Gold test set
sample = X_eval.iloc[:5][feature_cols].to_dict(orient="records")

score_response = requests.post(
    f"{DATABRICKS_HOST}/serving-endpoints/smd-anomaly-detector-endpoint/invocations",
    headers={
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json"
    },
    json={"dataframe_records": sample}
)

predictions = score_response.json()
print("✅ Live endpoint predictions:")
print(json.dumps(predictions, indent=2))

# Show actual vs predicted
print("\nActual labels for these 5 rows:")
print(y_eval.iloc[:5].values)

✅ Live endpoint predictions:
{
  "predictions": [
    0,
    0,
    0,
    0,
    0
  ]
}

Actual labels for these 5 rows:
[0 0 0 0 0]


In [0]:
# Find 5 actual anomalies and score them
anomaly_samples = X_eval[y_eval == 1].iloc[:5][feature_cols].to_dict(orient="records")

score_response = requests.post(
    f"{DATABRICKS_HOST}/serving-endpoints/smd-anomaly-detector-endpoint/invocations",
    headers={
        "Authorization": f"Bearer {DATABRICKS_TOKEN}",
        "Content-Type": "application/json"
    },
    json={"dataframe_records": anomaly_samples}
)

predictions = score_response.json()
print("✅ Anomaly predictions:")
print(json.dumps(predictions, indent=2))
print("\nActual labels (should all be 1):")
print(y_eval[y_eval == 1].iloc[:5].values)

✅ Anomaly predictions:
{
  "predictions": [
    1,
    1,
    1,
    1,
    1
  ]
}

Actual labels (should all be 1):
[1 1 1 1 1]


In [0]:
import requests
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Rebuild all variables
test_pd = spark.table("workspace.default.smd_gold_test").toPandas()

feature_cols = [f"metric_{i}" for i in range(38)] + [
  "metric_0_mean_5", "metric_0_mean_15", "metric_0_mean_60",
  "metric_0_std_5",  "metric_0_std_15",  "metric_0_roc"
]

X_test = test_pd[feature_cols]
y_test = test_pd["is_anomaly"]

X_train_sup, X_eval, y_train_sup, y_eval = train_test_split(
    X_test, y_test, test_size=0.3, random_state=42, stratify=y_test
)

DATABRICKS_HOST = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

print(f"✅ All variables ready")
print(f"   X_train_sup: {X_train_sup.shape}")
print(f"   X_eval:      {X_eval.shape}")

✅ All variables ready
   X_train_sup: (495894, 44)
   X_eval:      (212526, 44)


In [0]:
%pip install evidently
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.8/581.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 148.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 131.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 99.7 MB/s eta 0:00:00
  Attempting uninstall: sniffio
    Found existing installation: sniffio 1.3.0
    Not uninstalling sniffio at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-2ba6f9a2-dcf8-40b7-8eb2-23d1e60529e9
    Can't uninstall 'sniffio'. No files were found to uninstall.
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2023.5.0
    Not uninstalling fsspec at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemer

In [0]:
import evidently
print(evidently.__version__)

0.7.21


In [0]:
import evidently
import pkgutil

# List all available modules
for importer, modname, ispkg in pkgutil.iter_modules(evidently.__path__):
    print(modname)

_pydantic_compat
_registry
_version
cli
core
descriptors
errors
future
generators
guardrails
legacy
llm
metrics
nbextension
presets
pydantic_utils
sdk
telemetry
tests
ui
utils
widgets


In [0]:
from evidently.sdk.models import BatchRunRequest
import evidently.presets as presets

# Check what presets are available
print(dir(presets))

---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
File <command-6766441126044872>, line 1
----> 1 from evidently.sdk.models import BatchRunRequest
      2 import evidently.presets as presets
      4 # Check what presets are available

ImportError: cannot import name 'BatchRunRequest' from 'evidently.sdk.models' (/local_disk0/.ephemeral_nfs/envs/pythonEnv-2ba6f9a2-dcf8-40b7-8eb2-23d1e60529e9/lib/python3.12/site-packages/evidently/sdk/models.py)

In [0]:
import evidently.presets as presets
print(dir(presets))

from evidently import core
print(dir(core))

['ClassificationDummyQuality', 'ClassificationPreset', 'ClassificationQuality', 'ClassificationQualityByLabel', 'DataDriftPreset', 'DataSummaryPreset', 'DatasetStats', 'RecsysPreset', 'RegressionDummyQuality', 'RegressionPreset', 'RegressionQuality', 'TextEvals', 'ValueStats', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'classification', 'dataset_stats', 'drift', 'recsys', 'regression']
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_utils', 'base_types', 'compare', 'container', 'datasets', 'metric_types', 'registries', 'report', 'serialization', 'tests']


In [0]:
from evidently.core.report import Report
from evidently.presets import DataDriftPreset
from evidently.core.datasets import Dataset

# Check Dataset signature
help(Dataset)

Help on class Dataset in module evidently.core.datasets:

class Dataset(builtins.object)
 |  Dataset object that wraps your data with metadata and data definition.
 |
 |  `Dataset` is the main data structure in Evidently. It wraps a `pandas.DataFrame`
 |  with additional metadata including:
 |  - `DataDefinition`: column types and roles mapping
 |  - Descriptors: computed row-level scores (for text/LLM data)
 |  - Metadata and tags: additional information about the dataset
 |
 |  You typically create a `Dataset` from a `pandas.DataFrame` using `Dataset.from_pandas()`.
 |  Use `Dataset` objects with `Report.run()` to perform evaluations.
 |
 |  **Documentation**: See [Data Definition Guide](https://docs.evidentlyai.com/docs/library/data_definition) for column mapping.
 |
 |  Create from pandas DataFrame:
 |  ```python
 |  from evidently import Dataset, DataDefinition
 |
 |  dataset = Dataset.from_pandas(
 |      source_df,
 |      data_definition=DataDefinition()
 |  )
 |  ```
 |
 |  Ad

In [0]:
from evidently import Dataset, DataDefinition
from evidently.presets import DataDriftPreset
from evidently.tests import TestReport

# Check what's available in this version
import evidently
print(dir(evidently))

---------------------------------------------------------------------------
ImportError                               Traceback (most recent call last)
File <command-6766441126044870>, line 3
      1 from evidently import Dataset, DataDefinition
      2 from evidently.presets import DataDriftPreset
----> 3 from evidently.tests import TestReport
      5 # Check what's available in this version
      6 import evidently

ImportError: cannot import name 'TestReport' from 'evidently.tests' (/local_disk0/.ephemeral_nfs/envs/pythonEnv-2ba6f9a2-dcf8-40b7-8eb2-23d1e60529e9/lib/python3.12/site-packages/evidently/tests/__init__.py)

In [0]:
from evidently.core.report import Report
from evidently.presets import DataDriftPreset
from evidently import Dataset, DataDefinition

# Reference = training data, Current = new incoming data
reference_data = X_train_sup.copy()
current_data   = X_eval.copy()

# Create Evidently datasets
reference = Dataset.from_pandas(
    reference_data,
    data_definition=DataDefinition()
)
current = Dataset.from_pandas(
    current_data,
    data_definition=DataDefinition()
)

# Run drift report
report = Report([DataDriftPreset()])
my_eval = report.run(reference_data=reference, current_data=current)

# Save HTML
my_eval.save_html("/tmp/drift_report.html")
print("✅ Drift report generated")
print(type(my_eval))

/databricks/python/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/databricks/python/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/databricks/python/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/databricks/python/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/databricks/python/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning:

invalid value encountered in divide

/databricks/python/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning:

invalid value encountered in divide

/databricks/python/lib/python3.12/site-packages/numpy/lib/_function_ba

✅ Drift report generated
<class 'evidently.core.report.Snapshot'>


In [0]:
import json

results = my_eval.dict()

# See the top level keys
print("Top level keys:", results.keys())

# See first metric structure
print("\nFirst metric:")
print(json.dumps(results["metrics"][0], indent=2, default=str)[:2000])

Top level keys: dict_keys(['metrics', 'tests'])

First metric:
{
  "id": "15e89f895b482f9b84ba7274ed18a106",
  "metric_name": "DriftedColumnsCount(drift_share=0.5)",
  "config": {
    "type": "evidently:metric_v2:DriftedColumnsCount",
    "drift_share": 0.5
  },
  "value": {
    "count": 0.0,
    "share": 0.0
  }
}


In [0]:
import mlflow

# Extract directly from value
results = my_eval.dict()

drifted_count = results["metrics"][0]["value"]["count"]
drift_share   = results["metrics"][0]["value"]["share"]
dataset_drift = drift_share > 0.3  # True if more than 30% columns drifted

print(f"Drifted columns:          {drifted_count}")
print(f"Share of drifted columns: {drift_share:.1%}")
print(f"Dataset drift detected:   {dataset_drift}")

# Log to MLflow
mlflow.set_experiment("/Users/nadhiya.ganesan93@gmail.com/smd-anomaly-detection")

with mlflow.start_run(run_name="drift_monitoring_baseline"):
    mlflow.log_metric("drift_share",   float(drift_share))
    mlflow.log_metric("dataset_drift", int(dataset_drift))
    mlflow.log_metric("drifted_cols",  float(drifted_count))
    mlflow.log_artifact("/tmp/drift_report.html")
    print("\n✅ Drift metrics logged to MLflow")

# Alert if drift exceeds threshold
DRIFT_THRESHOLD = 0.3

if drift_share > DRIFT_THRESHOLD:
    print(f"\n⚠️  DRIFT ALERT: {drift_share:.1%} drifted — retraining recommended")
else:
    print(f"\n✅ Drift within range ({drift_share:.1%} < {DRIFT_THRESHOLD:.0%})")

Drifted columns:          0.0
Share of drifted columns: 0.0%
Dataset drift detected:   False

✅ Drift metrics logged to MLflow

✅ Drift within range (0.0% < 30%)
